In [1]:
import pandas as pd
import numpy as np

df_09 = pd.read_csv(
    "../data/raw/online_retail_09_10.csv",
    encoding="ISO-8859-1"
)

df_10 = pd.read_csv(
    "../data/raw/online_retail_10_11.csv",
    encoding="ISO-8859-1"
)

print("2009-10:", df_09.shape)
print("2010-11:", df_10.shape)

2009-10: (525461, 8)
2010-11: (541910, 8)


In [2]:
print(df_09.columns.tolist())
print(df_10.columns.tolist())

['ï»¿InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']
['ï»¿InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']


In [3]:
df = pd.concat([df_09, df_10], ignore_index=True)

print("Combined dataset:", df.shape)

Combined dataset: (1067371, 8)


In [4]:
def clean_retail_data(data):
    data = data.copy()

    # Remove records without customer/product information
    data = data.dropna(subset=["CustomerID", "Description"])

    # Remove duplicate transactions
    data = data.drop_duplicates()

    # Remove cancelled invoices
    data = data[
        ~data["InvoiceNo"].astype(str).str.startswith("C")
    ]

    # Keep valid quantities and prices
    data = data[data["Quantity"] > 0]
    data = data[data["UnitPrice"] > 0]

    # Convert data types
    data["CustomerID"] = data["CustomerID"].astype(int)
    data["InvoiceDate"] = pd.to_datetime(data["InvoiceDate"])

    return data

In [6]:
print(df.columns.tolist())

['ï»¿InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']


In [7]:
print(df.shape)

(1067371, 8)


In [8]:
print(df.head())

  ï»¿InvoiceNo StockCode                          Description  Quantity  \
0       489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1       489434    79323P                   PINK CHERRY LIGHTS        12   
2       489434    79323W                  WHITE CHERRY LIGHTS        12   
3       489434     22041         RECORD FRAME 7" SINGLE SIZE         48   
4       489434     21232       STRAWBERRY CERAMIC TRINKET BOX        24   

      InvoiceDate  UnitPrice  CustomerID         Country  
0  12/1/2009 7:45       6.95     13085.0  United Kingdom  
1  12/1/2009 7:45       6.75     13085.0  United Kingdom  
2  12/1/2009 7:45       6.75     13085.0  United Kingdom  
3  12/1/2009 7:45       2.10     13085.0  United Kingdom  
4  12/1/2009 7:45       1.25     13085.0  United Kingdom  


In [9]:
df.columns = df.columns.str.replace("ï»¿", "", regex=False).str.strip()

print(df.columns.tolist())

['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']


In [10]:
df = clean_retail_data(df)

In [11]:
df["Revenue"] = df["Quantity"] * df["UnitPrice"]

In [12]:
df["Year"] = df["InvoiceDate"].dt.year
df["Month"] = df["InvoiceDate"].dt.month
df["Day"] = df["InvoiceDate"].dt.day
df["Hour"] = df["InvoiceDate"].dt.hour
df["DayOfWeek"] = df["InvoiceDate"].dt.dayofweek

In [13]:
df["UnitPrice"].describe()

count    779425.000000
mean          3.218488
std          29.676140
min           0.001000
25%           1.250000
50%           1.950000
75%           3.750000
max       10953.500000
Name: UnitPrice, dtype: float64

In [14]:
product_month = (
    df.groupby(
        [
            "StockCode",
            "Year",
            "Month"
        ]
    )
    .agg(
        AvgPrice=("UnitPrice", "mean"),
        TotalQuantity=("Quantity", "sum"),
        TotalRevenue=("Revenue", "sum"),
        Transactions=("InvoiceNo", "nunique"),
        Countries=("Country", "nunique")
    )
    .reset_index()
)

In [15]:
product_month.head()

,StockCode,Year,Month,AvgPrice,TotalQuantity,TotalRevenue,Transactions,Countries
0,10002,2009,12,0.850000,212,180.20,17,3
1,10002,2010,1,0.850000,289,245.65,15,2
2,10002,2010,2,0.850000,255,216.75,13,4
3,10002,2010,3,0.817500,633,479.81,15,3
4,10002,2010,4,0.819048,1129,838.75,21,4


In [16]:
product_month.shape

(60591, 8)

In [17]:
price_variation = (
    product_month.groupby("StockCode")["AvgPrice"]
    .nunique()
)

price_variation.describe()

count    4631.000000
mean        5.409847
std         5.188584
min         1.000000
25%         2.000000
50%         4.000000
75%         7.000000
max        25.000000
Name: AvgPrice, dtype: float64

In [18]:
product_month.to_csv(
    "../data/processed/product_month_panel.csv",
    index=False
)

In [19]:
price_variation.describe()

count    4631.000000
mean        5.409847
std         5.188584
min         1.000000
25%         2.000000
50%         4.000000
75%         7.000000
max        25.000000
Name: AvgPrice, dtype: float64

In [20]:
product_month.shape

(60591, 8)

In [21]:
product_month.head()

,StockCode,Year,Month,AvgPrice,TotalQuantity,TotalRevenue,Transactions,Countries
0,10002,2009,12,0.850000,212,180.20,17,3
1,10002,2010,1,0.850000,289,245.65,15,2
2,10002,2010,2,0.850000,255,216.75,13,4
3,10002,2010,3,0.817500,633,479.81,15,3
4,10002,2010,4,0.819048,1129,838.75,21,4


In [22]:
price_variation_counts = (
    product_month.groupby("StockCode")["AvgPrice"]
    .nunique()
)

print("Products with 1 price:",
      (price_variation_counts == 1).sum())

print("Products with 2+ prices:",
      (price_variation_counts >= 2).sum())

print("Products with 3+ prices:",
      (price_variation_counts >= 3).sum())

print("Products with 4+ prices:",
      (price_variation_counts >= 4).sum())

Products with 1 price: 963
Products with 2+ prices: 3668
Products with 3+ prices: 2967
Products with 4+ prices: 2349


In [23]:
print(
    "Products with 3+ prices:",
    round((price_variation_counts >= 3).mean() * 100, 2),
    "%"
)

Products with 3+ prices: 64.07 %


In [24]:
eligible_products = price_variation_counts[
    price_variation_counts >= 3
].index

model_df = product_month[
    product_month["StockCode"].isin(eligible_products)
].copy()

print("Modeling dataset shape:", model_df.shape)
print("Eligible products:", model_df["StockCode"].nunique())

Modeling dataset shape: (48732, 8)
Eligible products: 2967


In [25]:
model_df.head()

,StockCode,Year,Month,AvgPrice,TotalQuantity,TotalRevenue,Transactions,Countries
0,10002,2009,12,0.850000,212,180.20,17,3
1,10002,2010,1,0.850000,289,245.65,15,2
2,10002,2010,2,0.850000,255,216.75,13,4
3,10002,2010,3,0.817500,633,479.81,15,3
4,10002,2010,4,0.819048,1129,838.75,21,4


In [26]:
model_df.describe()

,Year,Month,AvgPrice,TotalQuantity,TotalRevenue,Transactions,Countries
count,48732.000000,48732.000000,48732.000000,48732.000000,48732.000000,48732.000000,48732.000000
mean,2010.451962,6.890708,3.472865,204.336493,340.019175,14.846631,2.115838
std,0.566422,3.510290,14.554030,583.227987,881.404828,20.657909,1.631936
min,2009.000000,1.000000,0.050000,1.000000,0.100000,1.000000,1.000000
25%,2010.000000,4.000000,0.973333,15.000000,30.000000,3.000000,1.000000
50%,2010.000000,7.000000,1.890000,60.000000,103.145000,8.000000,1.000000
75%,2011.000000,10.000000,3.730000,203.000000,332.360000,19.000000,3.000000
max,2011.000000,12.000000,1599.260000,74215.000000,77183.600000,437.000000,17.000000


In [27]:
model_df.isnull().sum()

StockCode        0
Year             0
Month            0
AvgPrice         0
TotalQuantity    0
TotalRevenue     0
Transactions     0
Countries        0
dtype: int64

In [28]:
model_df = model_df.sort_values(
    ["StockCode", "Year", "Month"]
).reset_index(drop=True)

model_df.head()

,StockCode,Year,Month,AvgPrice,TotalQuantity,TotalRevenue,Transactions,Countries
0,10002,2009,12,0.850000,212,180.20,17,3
1,10002,2010,1,0.850000,289,245.65,15,2
2,10002,2010,2,0.850000,255,216.75,13,4
3,10002,2010,3,0.817500,633,479.81,15,3
4,10002,2010,4,0.819048,1129,838.75,21,4


In [29]:
model_df["LagQuantity_1"] = (
    model_df.groupby("StockCode")["TotalQuantity"]
    .shift(1)
)

In [30]:
model_df["LagPrice_1"] = (
    model_df.groupby("StockCode")["AvgPrice"]
    .shift(1)
)

In [31]:
model_df["RollingQuantity_3"] = (
    model_df.groupby("StockCode")["TotalQuantity"]
    .transform(
        lambda x: x.shift(1).rolling(3).mean()
    )
)

In [32]:
model_df["TimeIndex"] = (
    model_df["Year"] * 12 + model_df["Month"]
)

In [33]:
model_df["MonthSin"] = np.sin(
    2 * np.pi * model_df["Month"] / 12
)

model_df["MonthCos"] = np.cos(
    2 * np.pi * model_df["Month"] / 12
)

In [34]:
model_df[
    [
        "StockCode",
        "Year",
        "Month",
        "AvgPrice",
        "TotalQuantity",
        "LagQuantity_1",
        "LagPrice_1",
        "RollingQuantity_3",
        "MonthSin",
        "MonthCos"
    ]
].head(15)

,StockCode,Year,Month,AvgPrice,TotalQuantity,LagQuantity_1,LagPrice_1,RollingQuantity_3,MonthSin,MonthCos
0,10002,2009,12,0.850000,212,NaN,NaN,NaN,-2.449294e-16,1.000000e+00
1,10002,2010,1,0.850000,289,212.0,0.850000,NaN,5.000000e-01,8.660254e-01
2,10002,2010,2,0.850000,255,289.0,0.850000,NaN,8.660254e-01,5.000000e-01
3,10002,2010,3,0.817500,633,255.0,0.850000,252.000000,1.000000e+00,6.123234e-17
4,10002,2010,4,0.819048,1129,633.0,0.817500,392.333333,8.660254e-01,-5.000000e-01
5,10002,2010,5,0.832069,1409,1129.0,0.819048,672.333333,5.000000e-01,-8.660254e-01
6,10002,2010,6,0.845517,442,1409.0,0.832069,1057.000000,1.224647e-16,-1.000000e+00
7,10002,2010,7,0.844091,503,442.0,0.845517,993.333333,-5.000000e-01,-8.660254e-01
8,10002,2010,8,0.835556,581,503.0,0.844091,784.666667,-8.660254e-01,-5.000000e-01
9,10002,2010,9,0.850000,215,581.0,0.835556,508.666667,-1.000000e+00,-1.836970e-16


In [35]:
model_df = model_df.dropna(
    subset=[
        "LagQuantity_1",
        "LagPrice_1",
        "RollingQuantity_3"
    ]
).copy()

In [36]:
print("Modeling dataset after lag features:", model_df.shape)

Modeling dataset after lag features: (39831, 14)


In [37]:
print(model_df.isnull().sum())

StockCode            0
Year                 0
Month                0
AvgPrice             0
TotalQuantity        0
TotalRevenue         0
Transactions         0
Countries            0
LagQuantity_1        0
LagPrice_1           0
RollingQuantity_3    0
TimeIndex            0
MonthSin             0
MonthCos             0
dtype: int64
